In [3]:

import kagglehub
# Download latest version: path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
import pandas as pd
import numpy as np
import os
import sys
# Add parent directory to path to import modules from one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

# Grid search over k-center matching hyperparameters + OCT training
# Find best configuration based on PR-AUC
import kcenter_hyperparameter_search_global
import importlib
importlib.reload(kcenter_hyperparameter_search_global)
from kcenter_hyperparameter_search_global import run_global_kcenter_matching, build_undersampled_dataset
import model_IAI
importlib.reload(model_IAI)
from model_IAI import finetune_oct, evaluate_binary_oct
from itertools import product
# Train/test splits (similar to kcenter_hyperparameter_search_global.py)
from sklearn.model_selection import train_test_split
import utils_fraud
importlib.reload(utils_fraud)
from utils_fraud import *
TRAIN_TEST_SEED = 123
RESULTS_DIR = "./fraud_results"

# Load Kaggle Credit Card Fraud Detection dataset
df_fraud_raw = pd.read_csv("creditcard.csv")
# Separate features and target (target column is 'Class')
X_fraud = df_fraud_raw.drop('Class', axis=1)
y_fraud = df_fraud_raw['Class']

ModuleNotFoundError: No module named 'model_IAI'

In [3]:
# Explore credit card fraud dataset
if X_fraud is not None:
    print("=== CREDIT CARD FRAUD DATASET ===")
    print(f"Shape: {X_fraud.shape}")
    print(f"\nFeatures ({len(X_fraud.columns)}):")
    print(X_fraud.columns.tolist())
    print(f"\nFeature types:")
    print(X_fraud.dtypes.value_counts())
    print(f"\nTarget variable: 'Class'")
    print(f"Target distribution:\n{y_fraud.value_counts()}")
    print(f"\nClass imbalance ratio: {y_fraud.value_counts().min() / y_fraud.value_counts().max():.4f}")
    print(f"\nFirst few rows:")
    X_fraud.head()
    
    # Show some statistics
    print(f"\nDataset statistics:")
    print(f"  Total samples: {len(X_fraud):,}")
    print(f"  Fraud cases (1): {y_fraud.sum():,} ({y_fraud.mean()*100:.2f}%)")
    print(f"  Normal cases (0): {(y_fraud == 0).sum():,} ({(1-y_fraud.mean())*100:.2f}%)")
else:
    print("⚠️  Credit card fraud dataset not loaded. Please download from Kaggle first.")

=== CREDIT CARD FRAUD DATASET ===
Shape: (284807, 30)

Features (30):
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

Feature types:
float64    30
Name: count, dtype: int64

Target variable: 'Class'
Target distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Class imbalance ratio: 0.0017

First few rows:

Dataset statistics:
  Total samples: 284,807
  Fraud cases (1): 492 (0.17%)
  Normal cases (0): 284,315 (99.83%)


In [15]:
# Setup for credit card fraud dataset
if X_fraud is not None:
    df_fraud = prepare_dataset_for_kcenter(X_fraud, y_fraud, "credit_card_fraud")
    df_fraud,feature_cols_fraud, CAT_COLUMNS_FRAUD, TRUE_NUM_COLUMNS_FRAUD, BIN_COLUMNS_FRAUD = setup_feature_columns(df_fraud, drop_time=False, log1p_amount=False)
    train_fraud, val_fraud, test_fraud = create_train_test_split(df_fraud, random_state=TRAIN_TEST_SEED)
    display(train_fraud.columns)
else:
    print("\n⚠️  Skipping credit card fraud dataset splits (file not loaded)")



=== CREDIT_CARD_FRAUD DATASET PREPARED ===
Shape: (284807, 32)
Original target type: <class 'pandas.core.series.Series'>
Original target unique values: [np.int64(0), np.int64(1)]
Target distribution:
target
0    284315
1       492
Name: count, dtype: int64
Class imbalance ratio: 0.0017

=== COLUMN SETUP ===
Total features: 30
Categorical columns: 0
Binary columns: 0
True numeric columns: 30

=== DATA SPLITS ===
Train: 199,364 samples
  - Minority: 344
  - Majority: 199,020
Val: 42,721 samples
  - Minority: 74
  - Majority: 42,647
Test: 42,722 samples
  - Minority: 74
  - Majority: 42,648


Index(['ENROLID', 'target', 'Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7',
       'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17',
       'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27',
       'V28', 'Amount'],
      dtype='object')

In [16]:
# Precompute distances for credit card fraud dataset
h5_path_fraud, X_controls_fraud, majority_enrolids_fraud = precompute_case_control_distances(
    train_fraud, 'target', feature_cols_fraud, CAT_COLUMNS_FRAUD, TRUE_NUM_COLUMNS_FRAUD,
    'creditcard_fraud', seed=TRAIN_TEST_SEED
)

Cases (minority): 344
Controls (majority): 199,020
→ Building preprocessor w/ imputation:
   • Cat: impute(most_frequent) + OHE on: []
   • Num: impute(median) + scale on: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
Computing 199,020 x 344 = 68,462,880 distances
Output size: 273.9 MB (<class 'numpy.float32'>)


Computing distances: 100%|██████████| 200/200 [00:00<00:00, 1130.28it/s]

  Distance matrix shape: (199020, 344)
  Distance range: [0.178, 240.310]
  Distance mean: 24.092

Saving to HDF5: ./precomputed_distances/distances_creditcard_fraud_seed_123.h5


  ✓ Saved 241.0 MB

✓ Saved distances to: ./precomputed_distances/distances_creditcard_fraud_seed_123.h5


## OCT gridsearch for best kcenter config

AUC score: 0.904
PR-AUC (Average Precision): 0.608
Best MCC: 0.757 @ threshold=0.454579
Balanced (G-mean) recall: 0.689
Balanced (G-mean) specificity: 0.999
Number of leaves: 4

VS. BALANCED

AUC score: 0.911
PR-AUC (Average Precision): 0.494
Best MCC: 0.693 @ threshold=0.548562
Balanced (G-mean) recall: 0.824
Balanced (G-mean) specificity: 0.990
Number of leaves: 3

In [17]:
#h5_path_fraud = os.path.join(f"./precomputed_distances/distances_creditcard_fraud_seed_{TRAIN_TEST_SEED}_new.h5")

# OCT hyperparameters
OCT_DEPTHS = [7]
OCT_MINBUCKETS_FRAUD = [25] #,50,75
OCT_CPS = [0.01] #0.00001, 0.0001, 0.001, 

# Results directory
RESULTS_DIR = "./uci_experiments_results/matching_ratio_comparison"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Fixed hyperparameters for k-center matching
SEED_METHOD = "smart"
CASE_WEIGHTING = None
USE_ADAPTIVE_POOL = True

# Matching ratios to test
MATCHING_RATIOS = [1] #, 2, 3, 4, 5

print("="*80)
print("MATCHING RATIO COMPARISON: K-CENTER MATCHING + OCT TRAINING")
print("="*80)
print(f"\nFixed hyperparameters:")
print(f"  seed_method: {SEED_METHOD}")
print(f"  case_weighting: {CASE_WEIGHTING}")
print(f"  use_adaptive_pool: {USE_ADAPTIVE_POOL}")
print(f"\nMatching ratios to test: {MATCHING_RATIOS}")

# Import matplotlib for plotting
import matplotlib.pyplot as plt

# ============================================================================
# CREDIT CARD FRAUD DATASET
# ============================================================================
if 'X_fraud' in globals() and X_fraud is not None:
    print("\n" + "="*80)
    print("CREDIT CARD FRAUD DATASET - MATCHING RATIO COMPARISON")
    print("="*80)
    
    # Prepare validation and test sets
    X_val_fraud = val_fraud[feature_cols_fraud]
    y_val_fraud = val_fraud['target']
    X_test_fraud = test_fraud[feature_cols_fraud]
    y_test_fraud = test_fraud['target']
    
    # Use dataset-specific DNN directory to avoid conflicts
    dnn_dir_fraud = f"./precomputed_distances/global_dnn_fraud_seed_{TRAIN_TEST_SEED}"
    
    # Store all results
    all_results_fraud = []
    
    # Loop over matching ratios
    for matching_ratio in MATCHING_RATIOS:
        print(f"\n{'#'*80}")
        print(f"MATCHING RATIO: 1:{matching_ratio}")
        print(f"{'#'*80}")
        
        config_name = f"seed_{SEED_METHOD}_cw_{CASE_WEIGHTING}_pool_{USE_ADAPTIVE_POOL}_ratio_{matching_ratio}"
        
        # Run k-center matching
        matching_result = run_global_kcenter_matching(
            train_pd=train_fraud,
            target_col='target',
            feature_cols=feature_cols_fraud,
            pn_h5_path=h5_path_fraud,
            matching_ratio=matching_ratio,
            case_weighting=CASE_WEIGHTING,
            use_adaptive_pool=USE_ADAPTIVE_POOL,
            seed_method=SEED_METHOD,
            candidate_pool_size=None,
            CAT_COLUMNS=CAT_COLUMNS_FRAUD,
            TRUE_NUM_COLUMNS=TRUE_NUM_COLUMNS_FRAUD,
            COST_COLUMNS=None,
            dnn_out_dir=dnn_dir_fraud,
        )
        
        # Build undersampled dataset
        undersampled_fraud = build_undersampled_dataset(
            train_pd=train_fraud,
            matching_result=matching_result,
            target_col='target',
            matching_ratio=matching_ratio,
        )
        
        # Train OCT
        print(f"\nTraining OCT for ratio 1:{matching_ratio}...")
        balanced_model, balanced_params, _, preprocessor, feature_names = finetune_oct(
            X_train=undersampled_fraud[feature_cols_fraud],
            y_train=undersampled_fraud['target'],
            X_val=X_val_fraud,
            y_val=y_val_fraud,
            categorical_cols=CAT_COLUMNS_FRAUD,
            numeric_cols=TRUE_NUM_COLUMNS_FRAUD,
            depths=OCT_DEPTHS,
            minbuckets=OCT_MINBUCKETS_FRAUD,
            cps=OCT_CPS,
        )
        
        # Evaluate on TEST set (final evaluation)
        test_metrics = evaluate_binary_oct(
            balanced_model, X_test_fraud, y_test_fraud,
            preprocessor, feature_names,
            results_dir=RESULTS_DIR, save_suffix=f"fraud_ratio_{matching_ratio}",
            #X_val_df=X_val_fraud, y_val=y_val_fraud  # Use validation for threshold tuning
        )
        
        # Extract metrics
        test_auc = test_metrics.get('auc', 0) if isinstance(test_metrics.get('auc'), (int, float)) else 0
        test_pr_auc = test_metrics.get('pr_auc', 0) if isinstance(test_metrics.get('pr_auc'), (int, float)) else 0
        test_mcc = test_metrics.get('best_mcc', 0) if isinstance(test_metrics.get('best_mcc'), (int, float)) else 0
        
        # Store results
        result = {
            'matching_ratio': matching_ratio,
            'config_name': config_name,
            'seed_method': SEED_METHOD,
            'case_weighting': CASE_WEIGHTING,
            'use_adaptive_pool': USE_ADAPTIVE_POOL,
            'n_train_samples': len(undersampled_fraud),
            'n_train_minority': (undersampled_fraud['target'] == 1).sum(),
            'n_train_majority': (undersampled_fraud['target'] == 0).sum(),
            'test_auc': test_auc,
            'test_pr_auc': test_pr_auc,
            'test_mcc': test_mcc,
        }
        all_results_fraud.append(result)
        
        print(f"\n✓ Ratio 1:{matching_ratio} complete")
        print(f"   Test AUC: {test_auc:.4f}")
        print(f"   Test PR-AUC: {test_pr_auc:.4f}")
        print(f"   Test MCC: {test_mcc:.4f}")
        print(f"   Training samples: {len(undersampled_fraud):,} (minority: {(undersampled_fraud['target'] == 1).sum():,}, majority: {(undersampled_fraud['target'] == 0).sum():,})")



MATCHING RATIO COMPARISON: K-CENTER MATCHING + OCT TRAINING

Fixed hyperparameters:
  seed_method: smart
  case_weighting: None
  use_adaptive_pool: True

Matching ratios to test: [1]

CREDIT CARD FRAUD DATASET - MATCHING RATIO COMPARISON

################################################################################
MATCHING RATIO: 1:1
################################################################################

GLOBAL K-CENTER MATCHING CONFIGURATION:
  case_weighting: None
  use_adaptive_pool: True
  seed_method: smart
  matching_ratio: 1:1

Global Statistics:
  Cases (minority): 344
  Controls (majority): 199,020
  Ratio: 578.55:1

  Preprocessing:
    Features: 30
    Preprocessed shape: (199020, 30)

  Preparing control-control distances (global)...
    ⚙️  Computing global control-control distances (this is done once)...
[leaf global] computing d_nn for n=199,020 -> ~158.44 GB float32


leaf global d_nn: 100%|██████████| 266/266 [02:37<00:00,  1.69it/s]


[leaf global] saved:
  ./precomputed_distances/global_dnn_fraud_seed_123/leaf_global_dnn_matrix.npy
  ./precomputed_distances/global_dnn_fraud_seed_123/leaf_global_dnn_enrolids.npy
    ✓ Control-control distances computed and saved

  K-Center Configuration:
    M (candidate pool size): 99,510 / 199,020 (50.0%)
    Cases to match: 344
    Seed method: smart
    Adaptive pool: True
    Case weighting: None

  Running two-stage k-center matching (1:1)...
  Seed selection method: 'smart'
    Smart seed selected: index 64801 (mean dist to cases: 18.4375)
  Auto-computed tau (95th percentile of best distances): 24.7971
  Adaptive pool stopped at 19902 candidates (max cost: 9.4198)
    ✓ Matching complete!
    Cases matched: 344
    Total selected controls: 344 (unique: 344)
    Mean matching cost: 17.5729
    Sampling time: 13.14s
    Memory used: 4347.4 MB (Δ +587.3 MB)

BUILDING UNDERSAMPLED TRAINING DATASET

✓ Collected all minority samples: 344
✓ Collected selected majority samples: 344

# CREDIT CARD FRAUD DATASET - BASELINE

In [18]:
# Prepare validation and test sets
X_val_fraud = val_fraud[feature_cols_fraud]
y_val_fraud = val_fraud['target']
X_test_fraud = test_fraud[feature_cols_fraud]
y_test_fraud = test_fraud['target']

# Prepare original imbalanced training set
X_train_fraud_imbalanced = train_fraud[feature_cols_fraud]
y_train_fraud_imbalanced = train_fraud['target']


In [ ]:
import time
try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False
    print("Warning: psutil not available. Resource tracking will be limited.")

def format_time(seconds):
    """Format time in a readable way."""
    if seconds < 0.1:
        return f"{seconds:.4f}"
    elif seconds < 1.0:
        return f"{seconds:.3f}"
    else:
        return f"{seconds:.2f}"

def get_resource_usage():
    """Get current CPU and memory usage."""
    if PSUTIL_AVAILABLE:
        process = psutil.Process()
        memory_info = process.memory_info()
        return {
            'cpu_percent': process.cpu_percent(interval=0.1),
            'memory_mb': memory_info.rss / (1024 * 1024),  # RSS in MB
            'memory_percent': process.memory_percent(),
        }
    else:
        return {
            'cpu_percent': None,
            'memory_mb': None,
            'memory_percent': None,
        }

print(f"\nOriginal training set distribution:")
print(f"  Total samples: {len(X_train_fraud_imbalanced):,}")
print(f"  Minority (1): {y_train_fraud_imbalanced.sum():,} ({y_train_fraud_imbalanced.mean()*100:.2f}%)")
print(f"  Majority (0): {(y_train_fraud_imbalanced == 0).sum():,} ({(1-y_train_fraud_imbalanced.mean())*100:.2f}%)")
print(f"  Imbalance ratio: {(y_train_fraud_imbalanced == 0).sum() / y_train_fraud_imbalanced.sum():.2f}:1")

# Train OCT on imbalanced data
print(f"\n{'='*80}")
print("TRAINING OCT MODEL: Credit Card Fraud Dataset (Imbalanced Baseline)")
print(f"{'='*80}\n")
# Track resources before training
resources_before = get_resource_usage()
training_start_time = time.perf_counter()

baseline_model_fraud, baseline_params_fraud, _, baseline_preprocessor_fraud, baseline_feature_names_fraud = finetune_oct(
    X_train=X_train_fraud_imbalanced,
    y_train=y_train_fraud_imbalanced,
    X_val=X_val_fraud,
    y_val=y_val_fraud,
    categorical_cols=CAT_COLUMNS_FRAUD,
    numeric_cols=TRUE_NUM_COLUMNS_FRAUD,
    depths=[5,7],
    minbuckets=[50,100,150],
    cps=[0.00001,0.001],
)


## best : (5, 100, 1e-05) -- 0.904 AUC, 0.608 PR-AUC, 0.757 MCC
# best over depths 5,7,9: (7, 150, 1e-05) -- 0.876 AUC, PR-AUC 0.576, MCC 0.757
training_end_time = time.perf_counter()
training_time = training_end_time - training_start_time
resources_after_training = get_resource_usage()



Original training set distribution:
  Total samples: 199,364
  Minority (1): 344 (0.17%)
  Majority (0): 199,020 (99.83%)
  Imbalance ratio: 578.55:1

TRAINING OCT MODEL: Credit Card Fraud Dataset (Imbalanced Baseline)

Finetuning OCT with depths: [5, 7], minbuckets: [50, 100, 150], cps: [1e-05, 0.001], for best PR-AUC!!!
→ Building preprocessor:
   • OneHotEncoder on: []
   • StandardScaler on: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [22]:

# Evaluate
evaluation_start_time = time.perf_counter()
# Evaluate
baseline_metrics_fraud = evaluate_binary_oct(
    baseline_model_fraud, X_test_fraud, y_test_fraud, 
    baseline_preprocessor_fraud, baseline_feature_names_fraud,
    results_dir=RESULTS_DIR,
    X_val_df=X_val_fraud,
    y_val=y_val_fraud,
    save_suffix="fraud_baseline",
)
evaluation_end_time = time.perf_counter()
evaluation_time = evaluation_end_time - evaluation_start_time
resources_after_eval = get_resource_usage()

total_time = time.perf_counter() - training_start_time
print(f"   - OCT training time: {format_time(training_time)}s")
print(f"   - Evaluation time: {format_time(evaluation_time)}s")
print(f"   - Total time: {format_time(total_time)}s")
if PSUTIL_AVAILABLE:
    memory_delta_training = resources_after_training['memory_mb'] - resources_before['memory_mb']
    memory_delta_eval = resources_after_eval['memory_mb'] - resources_after_training['memory_mb']
    print(f"\n💻 Compute Resources:")
    print(f"   - Peak memory (training): {resources_after_training['memory_mb']:.1f} MB (Δ {memory_delta_training:+.1f} MB)")
    print(f"   - Peak memory (evaluation): {resources_after_eval['memory_mb']:.1f} MB (Δ {memory_delta_eval:+.1f} MB)")
    print(f"   - Peak CPU (training): {resources_after_training['cpu_percent']:.1f}%")
    print(f"   - Peak CPU (evaluation): {resources_after_eval['cpu_percent']:.1f}%")
print(f"{'='*80}\n")

print(f"\n✓ Credit card fraud dataset baseline OCT training complete!")
print(f"   Best params: {baseline_params_fraud}")
if isinstance(baseline_metrics_fraud, dict):
    print(f"   AUC: {baseline_metrics_fraud.get('auc', 'N/A'):.4f}" if isinstance(baseline_metrics_fraud.get('auc'), (int, float)) else f"   AUC: {baseline_metrics_fraud.get('auc', 'N/A')}")
    print(f"   PR-AUC: {baseline_metrics_fraud.get('pr_auc', 'N/A'):.4f}" if isinstance(baseline_metrics_fraud.get('pr_auc'), (int, float)) else f"   PR-AUC: {baseline_metrics_fraud.get('pr_auc', 'N/A')}")
    print(f"   Optimal F1: {baseline_metrics_fraud.get('optimal_f1', 'N/A'):.4f}" if isinstance(baseline_metrics_fraud.get('optimal_f1'), (int, float)) else f"   Optimal F1: {baseline_metrics_fraud.get('optimal_f1', 'N/A')}")
    print(f"   Best MCC: {baseline_metrics_fraud.get('best_mcc', 'N/A'):.4f}" if isinstance(baseline_metrics_fraud.get('best_mcc'), (int, float)) else f"   Best MCC: {baseline_metrics_fraud.get('best_mcc', 'N/A')}")


Test dataset for OCT application: 42,722 samples
✓ Predictions completed
Computing optimal thresholds on validation set (42,721 samples)
  Applied validation-set thresholds to test set for evaluation
✓ Saved OCT predictions to: ./uci_experiments_results/matching_ratio_comparison/predictions/oct_predictions_fraud_baseline.csv
✓ Saved split table (2 splits) to: ./uci_experiments_results/matching_ratio_comparison/oct_tree_fraud_baseline_splits.csv
AUC score: 0.844
PR-AUC (Average Precision): 0.608
Best MCC (test set, threshold from val): 0.776 @ threshold=0.441485
Sensitivity (Recall) @MCC*: 0.676
Specificity @MCC*: 1.000
Balanced (G-mean) recall (test set, threshold from val): 0.676
Balanced (G-mean) specificity (test set, threshold from val): 1.000
Number of leaves: 3
   - OCT training time: 369.25s
   - Evaluation time: 7.79s
   - Total time: 3045.55s

💻 Compute Resources:
   - Peak memory (training): 5198.2 MB (Δ +252.9 MB)
   - Peak memory (evaluation): 5220.8 MB (Δ +22.6 MB)
   - Pe

In [17]:
# Symmetric leaf evaluation: Compare baseline (vanilla) vs balanced (k-center) OCT models
import symmetric_excess_AUC
import importlib
importlib.reload(symmetric_excess_AUC)
from symmetric_excess_AUC import symmetric_leaf_evaluation_oct

print("="*80)
print("SYMMETRIC LEAF EVALUATION: Baseline vs Balanced OCT Models")
print("="*80)
# ============================================================================
# Paths to prediction files
vanilla_pred_path_fraud = f"./uci_experiments_results/predictions/oct_predictions_fraud_baseline.csv"
balanced_pred_path_fraud = f"./uci_experiments_results/predictions/seed_smart_cw_None_pool_True_predictions.csv"

# Check which files exist
if os.path.exists(vanilla_pred_path_fraud) and os.path.exists(balanced_pred_path_fraud):
    mv_path_fraud = vanilla_pred_path_fraud
    ms_path_fraud = balanced_pred_path_fraud
else:
    print("⚠️  Prediction files not found.")
    mv_path_fraud = None
    ms_path_fraud = None

if mv_path_fraud and ms_path_fraud:
    try:
        # Prepare y_test with index if available
        if 'test_fraud' in globals():
            y_test_fraud_series = pd.Series(test_fraud['target'].values, index=test_fraud['ENROLID'].values)
        else:
            y_test_fraud_series = pd.Series(y_test_fraud.values)
        
        subgroup_results_fraud = symmetric_leaf_evaluation_oct(
            mv_pred_path=mv_path_fraud,
            ms_pred_path=ms_path_fraud,
            y_test=y_test_fraud_series,
        )
        
        print("\n✓ Credit card fraud dataset symmetric evaluation complete!")
        print("\nSubgroup Results (CI):")
        display(subgroup_results_fraud['ci'])
        print("\nOverall Results:")
        display(subgroup_results_fraud['overall'])
        
    except Exception as e:
        print(f"✗ Error in fraud dataset symmetric evaluation: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Skipping fraud dataset symmetric evaluation (prediction files not found)")


SYMMETRIC LEAF EVALUATION: Baseline vs Balanced OCT Models

✓ Credit card fraud dataset symmetric evaluation complete!

Subgroup Results (CI):


{'excess_ROC_s|v': {'point': 0.23081364989461095,
  'lo': 0.11199243764084173,
  'hi': 0.3530526700080445,
  'se_boot': 0.06174409072764325,
  'n_eff': 1969,
  '_bootstrap_samples': [0.3528030688173037,
   0.1690602076076524,
   0.27797018179340716,
   0.34883560910717293,
   0.1965767285070457,
   0.27338980635520926,
   0.19542696557450334,
   0.1265117635291605,
   0.216536565029255,
   0.28547175177054684,
   0.2539462383377317,
   0.3290493676050591,
   0.17266751287300477,
   0.2036926807760141,
   0.3497984025804469,
   0.18955349620893003,
   0.2621487246000864,
   0.2355597599164927,
   0.2888894535200644,
   0.2454269000417847,
   0.2441286849643002,
   0.19903040526537108,
   0.23624034997426657,
   0.2308466977195831,
   0.2176105647573039,
   0.22846565643483152,
   0.31996298941627166,
   0.32972832782208383,
   0.20080395767485198,
   0.24432476407419457,
   0.2824850679567596,
   0.1512310114256038,
   0.20875475517993203,
   0.3521215490087086,
   0.180227875116175,
  


Overall Results:


{'M_v_global_ROC': 0.8760320182309491,
 'M_v_global_PR': 0.5761722799483503,
 'M_v_global_MCC': 0.7573352410673753,
 'M_s_global_ROC': 0.898059444503592,
 'M_s_global_PR': 0.6472099414826382,
 'M_s_global_MCC': 0.7552185582462523,
 'n_test': 42722,
 'prevalence': 0.0017321286456626562}

In [16]:
# Bootstrap p-value comparison: Undersampled model vs Baseline model
# Compare metrics from undersampled training (grid search best) vs full training set (baseline)

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef
from symmetric_excess_AUC import bootstrap_ci_global_metrics, _bootstrap_pvalue, _safe_mcc

print("="*80)
print("BOOTSTRAP P-VALUE COMPARISON: Undersampled vs Baseline Model")
print("="*80)

# Paths to prediction files
vanilla_pred_path_fraud = f"./uci_experiments_results/predictions/oct_predictions_fraud_baseline.csv"
balanced_pred_path_fraud = f"./uci_experiments_results/predictions/seed_smart_cw_None_pool_True_predictions.csv"

# Check if prediction files exist
if os.path.exists(vanilla_pred_path_fraud) and os.path.exists(balanced_pred_path_fraud):
    print("\nLoading predictions from CSV files...")
    
    # Load prediction files
    baseline_pred_df = pd.read_csv(vanilla_pred_path_fraud)
    balanced_pred_df = pd.read_csv(balanced_pred_path_fraud)
    
    print(f"  Baseline predictions: {len(baseline_pred_df):,} rows")
    print(f"  Balanced predictions: {len(balanced_pred_df):,} rows")
    
    # Check required columns
    if 'predicted_proba' not in baseline_pred_df.columns:
        raise ValueError(f"Baseline predictions file missing 'predicted_proba' column. Columns: {baseline_pred_df.columns.tolist()}")
    if 'predicted_proba' not in balanced_pred_df.columns:
        raise ValueError(f"Balanced predictions file missing 'predicted_proba' column. Columns: {balanced_pred_df.columns.tolist()}")
    
    # Extract predictions and align with test set
    # If files have ENROLID, align by ENROLID; otherwise use row order
    if 'ENROLID' in baseline_pred_df.columns and 'ENROLID' in balanced_pred_df.columns:
        print("  Aligning predictions by ENROLID...")
        # Align by ENROLID
        common_enrolids = set(baseline_pred_df['ENROLID']).intersection(set(balanced_pred_df['ENROLID']))
        if len(common_enrolids) == 0:
            raise ValueError("No common ENROLIDs found between prediction files")
        
        # Also need to align with test set
        if 'test_fraud' in globals() and 'ENROLID' in test_fraud.columns:
            test_enrolids = set(test_fraud['ENROLID'])
            common_enrolids = common_enrolids.intersection(test_enrolids)
            if len(common_enrolids) == 0:
                raise ValueError("No common ENROLIDs between predictions and test set")
            
            # Align all three
            baseline_pred_df = baseline_pred_df[baseline_pred_df['ENROLID'].isin(common_enrolids)].set_index('ENROLID')
            balanced_pred_df = balanced_pred_df[balanced_pred_df['ENROLID'].isin(common_enrolids)].set_index('ENROLID')
            test_aligned = test_fraud[test_fraud['ENROLID'].isin(common_enrolids)].set_index('ENROLID')
            
            # Sort by ENROLID for consistent ordering
            baseline_pred_df = baseline_pred_df.sort_index()
            balanced_pred_df = balanced_pred_df.sort_index()
            test_aligned = test_aligned.sort_index()
            
            y_test_pred_baseline = baseline_pred_df['predicted_proba'].values
            y_test_pred_balanced = balanced_pred_df['predicted_proba'].values
            y_test_array = test_aligned['target'].values
            
            print(f"  Aligned {len(common_enrolids):,} samples by ENROLID")
        else:
            # Align predictions by ENROLID only
            baseline_pred_df = baseline_pred_df[baseline_pred_df['ENROLID'].isin(common_enrolids)].set_index('ENROLID').sort_index()
            balanced_pred_df = balanced_pred_df[balanced_pred_df['ENROLID'].isin(common_enrolids)].set_index('ENROLID').sort_index()
            
            y_test_pred_baseline = baseline_pred_df['predicted_proba'].values
            y_test_pred_balanced = balanced_pred_df['predicted_proba'].values
            
            # Try to get y_test from test_fraud if available
            if 'test_fraud' in globals() and 'ENROLID' in test_fraud.columns:
                test_aligned = test_fraud[test_fraud['ENROLID'].isin(common_enrolids)].set_index('ENROLID').sort_index()
                y_test_array = test_aligned['target'].values
            elif 'y_test_fraud' in globals():
                # Use y_test_fraud if available (may need alignment)
                y_test_array = np.asarray(y_test_fraud).astype(int)[:len(y_test_pred_baseline)]
            else:
                raise ValueError("Cannot determine test labels. Need test_fraud or y_test_fraud.")
    else:
        # No ENROLID - align by row order (assumes same order)
        print("  Aligning predictions by row order (no ENROLID found)...")
        min_len = min(len(baseline_pred_df), len(balanced_pred_df))
        if 'y_test_fraud' in globals():
            min_len = min(min_len, len(y_test_fraud))
        
        y_test_pred_baseline = baseline_pred_df['predicted_proba'].values[:min_len]
        y_test_pred_balanced = balanced_pred_df['predicted_proba'].values[:min_len]
        
        if 'y_test_fraud' in globals():
            y_test_array = np.asarray(y_test_fraud).astype(int)[:min_len]
        else:
            raise ValueError("Cannot determine test labels. Need y_test_fraud.")
        
        print(f"  Using first {min_len:,} rows (assuming same order)")
    
    # Convert to numpy arrays
    p_baseline = np.asarray(y_test_pred_baseline).astype(float)
    p_balanced = np.asarray(y_test_pred_balanced).astype(float)
    y_test_array = np.asarray(y_test_array).astype(int)

   # Compute point estimates
    baseline_roc = roc_auc_score(y_test_array, p_baseline)
    balanced_roc = roc_auc_score(y_test_array, p_balanced)
    baseline_pr = average_precision_score(y_test_array, p_baseline)
    balanced_pr = average_precision_score(y_test_array, p_balanced)
    baseline_mcc = _safe_mcc(y_test_array, p_baseline)
    balanced_mcc = _safe_mcc(y_test_array, p_balanced)
    
    print(f"\nPoint Estimates (Test Set):")
    print(f"  Baseline (full training):")
    print(f"    ROC-AUC: {baseline_roc:.4f}")
    print(f"    PR-AUC: {baseline_pr:.4f}")
    print(f"    MCC: {baseline_mcc:.4f}")
    print(f"  Undersampled (best from grid search):")
    print(f"    ROC-AUC: {balanced_roc:.4f}")
    print(f"    PR-AUC: {balanced_pr:.4f}")
    print(f"    MCC: {balanced_mcc:.4f}")
    print(f"  Differences (Undersampled - Baseline):")
    print(f"    ROC-AUC: {balanced_roc - baseline_roc:+.4f}")
    print(f"    PR-AUC: {balanced_pr - baseline_pr:+.4f}")
    print(f"    MCC: {balanced_mcc - baseline_mcc:+.4f}")
    
    # Bootstrap comparison
    print(f"\n{'='*80}")
    print("BOOTSTRAPPING (B=2000, paired bootstrap)")
    print(f"{'='*80}")
    print("Computing bootstrap CIs and p-values...")
    
    bootstrap_results = bootstrap_ci_global_metrics(
        y=y_test_array,
        pv=p_baseline,  # Baseline predictions
        ps=p_balanced,  # Undersampled predictions
        B=2000,
        alpha=0.05,
        rng=42
    )
    
    # Extract bootstrap samples for p-values
    roc_diff_samples = bootstrap_results["_bootstrap_samples"]["roc_diff"]
    pr_diff_samples = bootstrap_results["_bootstrap_samples"]["pr_diff"]
    mcc_diff_samples = bootstrap_results["_bootstrap_samples"].get("mcc_diff", [])
    
    # Compute p-values (one-sided: H1: undersampled > baseline)
    pvalues = {}
    pvalues["ROC_AUC_diff"] = _bootstrap_pvalue(
        roc_diff_samples, null_value=0.0, alternative="greater"
    )
    pvalues["PR_AUC_diff"] = _bootstrap_pvalue(
        pr_diff_samples, null_value=0.0, alternative="greater"
    )
    if len(mcc_diff_samples) > 0:
        pvalues["MCC_diff"] = _bootstrap_pvalue(
            mcc_diff_samples, null_value=0.0, alternative="greater"
        )
    else:
        pvalues["MCC_diff"] = np.nan
    
    # Display results
    print(f"\n{'='*80}")
    print("BOOTSTRAP RESULTS")
    print(f"{'='*80}")
    
    print(f"\n📊 Confidence Intervals (95%):")
    print(f"\n  ROC-AUC:")
    print(f"    Baseline: {bootstrap_results['global_ROC_Mv']['point']:.4f} [{bootstrap_results['global_ROC_Mv']['lo']:.4f}, {bootstrap_results['global_ROC_Mv']['hi']:.4f}]")
    print(f"    Undersampled: {bootstrap_results['global_ROC_Ms']['point']:.4f} [{bootstrap_results['global_ROC_Ms']['lo']:.4f}, {bootstrap_results['global_ROC_Ms']['hi']:.4f}]")
    print(f"    Difference: {bootstrap_results['global_ROC_diff_Ms_minus_Mv']['point']:+.4f} [{bootstrap_results['global_ROC_diff_Ms_minus_Mv']['lo']:.4f}, {bootstrap_results['global_ROC_diff_Ms_minus_Mv']['hi']:.4f}]")
    
    print(f"\n  PR-AUC:")
    print(f"    Baseline: {bootstrap_results['global_PR_Mv']['point']:.4f} [{bootstrap_results['global_PR_Mv']['lo']:.4f}, {bootstrap_results['global_PR_Mv']['hi']:.4f}]")
    print(f"    Undersampled: {bootstrap_results['global_PR_Ms']['point']:.4f} [{bootstrap_results['global_PR_Ms']['lo']:.4f}, {bootstrap_results['global_PR_Ms']['hi']:.4f}]")
    print(f"    Difference: {bootstrap_results['global_PR_diff_Ms_minus_Mv']['point']:+.4f} [{bootstrap_results['global_PR_diff_Ms_minus_Mv']['lo']:.4f}, {bootstrap_results['global_PR_diff_Ms_minus_Mv']['hi']:.4f}]")
    
    if len(mcc_diff_samples) > 0:
        print(f"\n  MCC:")
        print(f"    Baseline: {bootstrap_results['global_MCC_Mv']['point']:.4f} [{bootstrap_results['global_MCC_Mv']['lo']:.4f}, {bootstrap_results['global_MCC_Mv']['hi']:.4f}]")
        print(f"    Undersampled: {bootstrap_results['global_MCC_Ms']['point']:.4f} [{bootstrap_results['global_MCC_Ms']['lo']:.4f}, {bootstrap_results['global_MCC_Ms']['hi']:.4f}]")
        print(f"    Difference: {bootstrap_results['global_MCC_diff_Ms_minus_Mv']['point']:+.4f} [{bootstrap_results['global_MCC_diff_Ms_minus_Mv']['lo']:.4f}, {bootstrap_results['global_MCC_diff_Ms_minus_Mv']['hi']:.4f}]")
    
    print(f"\n📈 P-Values (One-sided test: H1: Undersampled > Baseline):")
    pvals_df = pd.DataFrame([
        {"Metric": "ROC-AUC difference", 
         "Difference": f"{balanced_roc - baseline_roc:+.4f}",
         "P-value": pvalues["ROC_AUC_diff"]},
        {"Metric": "PR-AUC difference", 
         "Difference": f"{balanced_pr - baseline_pr:+.4f}",
         "P-value": pvalues["PR_AUC_diff"]},
        {"Metric": "MCC difference", 
         "Difference": f"{balanced_mcc - baseline_mcc:+.4f}",
         "P-value": pvalues["MCC_diff"]},
    ])
    display(pvals_df)
    
    print(f"\n💡 Interpretation:")
    print(f"  P-value < 0.05: Undersampled model is significantly better (α=0.05)")
    print(f"  P-value < 0.01: Undersampled model is significantly better (α=0.01)")
    print(f"  P-value < 0.001: Undersampled model is significantly better (α=0.001)")
    
    # Store results
    comparison_results = {
        'point_estimates': {
            'baseline_roc': baseline_roc,
            'baseline_pr': baseline_pr,
            'baseline_mcc': baseline_mcc,
            'balanced_roc': balanced_roc,
            'balanced_pr': balanced_pr,
            'balanced_mcc': balanced_mcc,
        },
        'bootstrap_ci': bootstrap_results,
        'pvalues': pvalues,
    }
    
    print(f"\n✓ Results stored in 'comparison_results' variable")
    print(f"{'='*80}\n")
    
else:
    print("\n⚠️  Models not found. Please ensure both baseline_model_fraud and balanced_model_fraud are trained.")
    print("   Run the baseline training cell and grid search cell first.")

BOOTSTRAP P-VALUE COMPARISON: Undersampled vs Baseline Model

Loading predictions from CSV files...
  Baseline predictions: 42,722 rows
  Balanced predictions: 42,722 rows
  Aligning predictions by row order (no ENROLID found)...
  Using first 42,722 rows (assuming same order)

Point Estimates (Test Set):
  Baseline (full training):
    ROC-AUC: 0.8760
    PR-AUC: 0.5762
    MCC: 0.7573
  Undersampled (best from grid search):
    ROC-AUC: 0.8981
    PR-AUC: 0.6472
    MCC: 0.7552
  Differences (Undersampled - Baseline):
    ROC-AUC: +0.0220
    PR-AUC: +0.0710
    MCC: -0.0021

BOOTSTRAPPING (B=2000, paired bootstrap)
Computing bootstrap CIs and p-values...

BOOTSTRAP RESULTS

📊 Confidence Intervals (95%):

  ROC-AUC:
    Baseline: 0.8749 [0.8198, 0.9226]
    Undersampled: 0.8978 [0.8470, 0.9422]
    Difference: +0.0229 [-0.0289, 0.0743]

  PR-AUC:
    Baseline: 0.5754 [0.4556, 0.6994]
    Undersampled: 0.6492 [0.5245, 0.7629]
    Difference: +0.0738 [-0.0219, 0.1678]

  MCC:
    Basel

,Metric,Difference,P-value
0,ROC-AUC difference,+0.0220,0.1995
1,PR-AUC difference,+0.0710,0.0715
2,MCC difference,-0.0021,0.4765



💡 Interpretation:
  P-value < 0.05: Undersampled model is significantly better (α=0.05)
  P-value < 0.01: Undersampled model is significantly better (α=0.01)
  P-value < 0.001: Undersampled model is significantly better (α=0.001)

✓ Results stored in 'comparison_results' variable

